In [ ]:
# Práctica 2026/27 — Clasificación de textos en Esperanto
## Tema: Religiones — Judaismo · Cristanismo · Islam
"""
**Equipo:** Marco Antonio Benali, Lucía Monteagudo y Sofía Madrazo
**Asignatura:** Minería de Textos
"""

In [ ]:
import re
import json
import time
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dataclasses import dataclass

from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

RANDOM_STATE = 42
DATASET_CSV  = "dataset_esperanto_religio.csv"
DATASET_URL  = "https://raw.githubusercontent.com/TU_USUARIO/TU_REPO/main/dataset_esperanto_religio.csv"
RESULTS_JSON = "results_religio.json"
GENERAR_DATASET = True
MAX_FEATURES    = 3000
WIKI_API = "https://eo.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "CUNEF-MineriaTextos/1.0 (practica academica)"}

In [ ]:
def clean_paragraph(p: str) -> str:
    p = re.sub(r"<[^>]+>", " ", p)
    p = re.sub(r"\[[^\]]*\]", " ", p)
    p = re.sub(r"https?://\S+", " ", p)
    p = p.replace(";", ",")
    p = re.sub(r"\s+", " ", p).strip()
    return p

def paragraphs_from_html(html: str, min_len: int = 120):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup.find_all(["sup", "style", "script", "figure", "table"]):
        tag.decompose()
    out = []
    for p in soup.find_all("p"):
        text = clean_paragraph(p.get_text(" ", strip=True))
        if len(text) >= min_len:
            out.append(text)
    return out

def wiki_search(query: str, limit: int = 10):
    params = {
        "action": "query", "list": "search",
        "srsearch": query, "srlimit": limit,
        "format": "json", "utf8": 1,
    }
    r = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return [x["title"] for x in r.json().get("query", {}).get("search", [])]

def wiki_article_html(title: str):
    params = {
        "action": "parse", "page": title,
        "prop": "text|links", "format": "json", "redirects": 1,
    }
    r = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    data = r.json()
    if "parse" not in data:
        return "", []
    html  = data["parse"].get("text", {}).get("*", "")
    links = [l["*"] for l in data["parse"].get("links", []) if l.get("ns") == 0]
    return html, links

def crawl_topic(seed_titles, keywords, max_articles=8):
    visited, queue, articles = set(), list(seed_titles), {}
    while queue and len(visited) < max_articles:
        title = queue.pop(0)
        if title in visited:
            continue
        visited.add(title)
        try:
            html, links = wiki_article_html(title)
            articles[title] = html
            for link in links:
                if link in visited:
                    continue
                if any(kw.lower() in link.lower() for kw in keywords):
                    queue.append(link)
            time.sleep(0.2)
        except Exception as e:
            print(f"  Error con {title}: {e}")
    return articles

CATEGORY_SEEDS = {
    "judaismo":   (["Judaismo", "Torao", "Sinagogo", "Rabeno", "Talmudo",
                    "Hebrea lingvo", "Holokaŭsto"],
                   ["jud", "hebre", "sinagog", "tora", "raben", "talmud"]),
    "kristanismo":(["Kristanismo", "Jesuo", "Biblio", "Preĝejo", "Katolikismo",
                    "Protestantismo", "Ortodoksismo", "Papo"],
                   ["kristan", "krist", "preĝej", "bibl", "jesu", "eklez", "papo"]),
    "islamo":     (["Islamo", "Korano", "Mohamedo", "Moskeo", "Ramadano",
                    "Sunismo", "Ŝijaismo", "Hajĝo"],
                   ["islam", "koran", "mohamed", "moske", "ramadan", "sunism", "ŝija"]),
}

def build_dataset(min_len=120):
    rows = []
    for label, (seeds, kws) in CATEGORY_SEEDS.items():
        print(f"[{label}] crawling {len(seeds)} semillas…")
        articles = crawl_topic(seeds, kws, max_articles=10)
        for title, html in articles.items():
            for p in paragraphs_from_html(html, min_len=min_len):
                rows.append({"text": p, "class": label})
    df = pd.DataFrame(rows).drop_duplicates(subset=["text"]).reset_index(drop=True)
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    return df

if GENERAR_DATASET:
    df = build_dataset()
    df.to_csv(DATASET_CSV, sep=";", index=False, encoding="utf-8")
else:
    try:
        df = pd.read_csv(DATASET_URL, sep=";", encoding="utf-8")
    except Exception:
        try:
            df = pd.read_csv(DATASET_CSV, sep=";", encoding="utf-8")
        except FileNotFoundError:
            df = build_dataset()
            df.to_csv(DATASET_CSV, sep=";", index=False, encoding="utf-8")

print("Shape:", df.shape)
print(df["class"].value_counts())
print(df.head(3))

In [ ]:
print(df.info())
print("\nColumnas:", df.columns.tolist())
print("Tipos:", df.dtypes.to_dict())

print("\nDistribución de clases:")
print(df["class"].value_counts())
print("\nBalance (%):")
print((df["class"].value_counts(normalize=True) * 100).round(2))

df["text_len"] = df["text"].str.len()
df["n_words"]  = df["text"].apply(lambda t: len(t.split()))
df["avg_word_len"] = df["text"].apply(lambda t: np.mean([len(w) for w in t.split()]))
df["len_bin"] = pd.cut(df["text_len"], bins=4,
                       labels=["very_short", "short", "long", "very_long"])

print("\nEstadísticas por clase:")
print(df.groupby("class")[["text_len", "n_words", "avg_word_len"]].describe().round(2))

print("\nCorrelación:")
print(df.select_dtypes(include="number").corr().round(3))

y_dummies = pd.get_dummies(df["class"], prefix="cls")
print("\nEtiquetas en One-Hot (primeras 3):")
print(y_dummies.head(3))

In [ ]:
STOPWORDS = set("""
la kaj aŭ sed ĉar ke se dum ĝis ĉe en sur sub super apud inter kontraŭ malgraŭ
per por pro pri de al el post antaŭ ekster tra trans sen kun
mi vi li ŝi ĝi ni ili oni sin si mia via lia ŝia ĝia nia ilia
miaj viaj liaj ŝiaj ĝiaj niaj iliaj
tiu tio tia tie ĉi tiuj ĉiuj ĉiu iu iuj neniu neniuj
kiu kio kia kie kiam kial kiel kiom
ne jes ĉu ajn do ja nur ankoraŭ tre pli plej malpli malplej tro tamen kvankam
esti estas estis estos estus
""".split())

PREPOSITIONS = set("en sur sub super apud inter kontraŭ malgraŭ per por pro pri de al el post antaŭ ekster tra trans sen kun ĝis ĉe".split())
PRONOUNS     = set("mi vi li ŝi ĝi ni ili oni sin si mia via lia ŝia ĝia nia ilia min vin lin ŝin ĝin nin ilin".split())
CONJUNCTIONS = set("kaj aŭ sed ĉar ke se dum ĝis kvankam tamen".split())
DETERMINERS  = set("la tiu tio tia tiuj ĉiuj ĉiu iu iuj neniu neniuj".split())
PARTICLES    = set("ne jes ĉu ajn do ja nur ankoraŭ tre pli plej malpli malplej tro".split())
PUNCT = set(".,;:!?()[]{}'\"«»…-–—/")

VERB_ENDINGS = ["as", "is", "os", "us", "u", "i"]
PART_ENDINGS = ["anta", "inta", "onta", "ata", "ita", "ota",
                "ante", "inte", "onte", "ate", "ite", "ote"]
NOUN_ENDINGS = ["ojn", "oj", "on", "o"]
ADJ_ENDINGS  = ["aj", "an", "a"]

def strip_punct(w: str) -> str:
    return w.strip(".,;:!?()[]{}'\"«»…-–—/")

def decompose(word: str) -> dict:
    w = strip_punct(word.lower())
    if not w:
        return {"prefix": "", "lexeme": "", "suffix": "", "ending": "", "raw": word}

    ending = ""
    rest = w
    for e in sorted(VERB_ENDINGS + PART_ENDINGS + NOUN_ENDINGS + ADJ_ENDINGS + ["e"],
                    key=len, reverse=True):
        if w.endswith(e) and len(w) > len(e) + 1:
            ending, rest = e, w[:-len(e)]
            break

    prefixes = ["kontraŭ", "mal", "re", "ge", "bo", "pra", "dis", "ek", "el", "en",
                "kun", "sen", "super", "sub", "sur", "trans", "mis"]
    prefix = ""
    for p in sorted(prefixes, key=len, reverse=True):
        if rest.startswith(p) and len(rest) > len(p) + 1:
            prefix, rest = p, rest[len(p):]
            break

    suffixes = ["eg", "et", "in", "id", "ul", "ar", "ej", "il", "ist", "ism",
                "ebl", "ind", "em", "end", "ad", "aĵ", "ec", "ig", "iĝ"]
    suffix = ""
    for s in sorted(suffixes, key=len, reverse=True):
        if rest.endswith(s) and len(rest) > len(s) + 1:
            suffix, rest = s, rest[:-len(s)]
            break

    return {"prefix": prefix, "lexeme": rest, "suffix": suffix,
            "ending": ending, "raw": word}

def lemma(word: str) -> str:
    w = strip_punct(word.lower())
    if not w:
        return ""
    if w in STOPWORDS or w in PREPOSITIONS or w in PRONOUNS \
            or w in CONJUNCTIONS or w in DETERMINERS or w in PARTICLES:
        return w

    for e in VERB_ENDINGS:
        if w.endswith(e) and len(w) > len(e) + 1:
            return w[:-len(e)] + "i"
    for e in PART_ENDINGS:
        if w.endswith(e) and len(w) > len(e) + 1:
            return w[:-len(e)] + "i"
    if w.endswith("ojn") and len(w) > 3: return w[:-3] + "o"
    if w.endswith("oj")  and len(w) > 2: return w[:-2] + "o"
    if w.endswith("on")  and len(w) > 2: return w[:-2] + "o"
    if w.endswith("aj")  and len(w) > 2: return w[:-2] + "a"
    if w.endswith("an")  and len(w) > 2: return w[:-2] + "a"
    if len(w) > 1 and w[-1] in "oaei":
        return w
    return w

def pos_tag(word: str) -> str:
    if word and all(ch in PUNCT for ch in word):
        return "PUNCT"
    w = strip_punct(word.lower())
    if not w:
        return "SPACE"
    if w in DETERMINERS or w == "la":     return "DET"
    if w in PRONOUNS:                     return "PRON"
    if w in PREPOSITIONS:                 return "ADP"
    if w in CONJUNCTIONS:                 return "CCONJ" if w in {"kaj","aŭ","sed"} else "SCONJ"
    if w in PARTICLES:                    return "PART"
    if re.fullmatch(r"\d+([.,]\d+)?", w): return "NUM"
    if w.endswith(tuple(VERB_ENDINGS)):   return "VERB"
    if w.endswith(tuple(PART_ENDINGS)):   return "VERB"
    if w.endswith(("ojn","oj","on","o")): return "NOUN"
    if w.endswith(("aj","an","a")):       return "ADJ"
    if w.endswith("e"):                   return "ADV"
    return "X"

@dataclass
class Token:
    text: str
    lemma_: str
    pos_: str
    is_stop: bool
    idx: int

class Doc:
    def __init__(self, text: str):
        self.text = text
        self.tokens = []
        for i, m in enumerate(re.finditer(r"\w+|[^\w\s]", text, flags=re.UNICODE)):
            raw = m.group(0)
            self.tokens.append(Token(
                text=raw, lemma_=lemma(raw), pos_=pos_tag(raw),
                is_stop=lemma(raw) in STOPWORDS, idx=i,
            ))
    def __iter__(self):  return iter(self.tokens)
    def __len__(self):   return len(self.tokens)
    @property
    def sents(self):
        return [Doc(p) for p in re.split(r"(?<=[.!?])\s+", self.text) if p.strip()]

class EsperantoNLP:
    def __call__(self, text): return Doc(text)
    def preprocess(self, text, remove_stop=True, lemmatize=True):
        out = []
        for t in Doc(text):
            if t.pos_ == "PUNCT":
                continue
            if remove_stop and t.is_stop:
                continue
            out.append(t.lemma_ if lemmatize else t.text.lower())
        return out

nlp = EsperantoNLP()

demo = "La judoj preĝas en la sinagogo kaj la kristanoj en la preĝejo."
for t in nlp(demo):
    if t.pos_ != "PUNCT":
        print(f"{t.text:15} lemma={t.lemma_:12} pos={t.pos_:6} stop={t.is_stop}")

print("\nDescomposición morfológica:")
for w in ["kontraŭjuda", "kristanojn", "islamaĵoj", "preĝejoj"]:
    print(w, "→", decompose(w))

In [ ]:
X_text = df["text"].values
y = df["class"].values

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    X_text, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y,
)

word_freq = {}
for t in X_train_txt:
    for w in nlp.preprocess(t):
        word_freq[w] = word_freq.get(w, 0) + 1

top_words = sorted(word_freq.items(), key=lambda x: -x[1])[:MAX_FEATURES]
vocab = [w for w, _ in top_words]
word_to_idx = {w: i for i, w in enumerate(vocab)}

print(f"Vocabulario: {len(vocab)} palabras")

def one_hot_mean(text):
    words = [w for w in nlp.preprocess(text) if w in word_to_idx]
    if not words:
        return np.zeros(len(vocab), dtype=np.float32)
    v = np.zeros(len(vocab), dtype=np.float32)
    for w in words:
        v[word_to_idx[w]] = 1.0
    return v / len(words)

X_train_oh = np.array([one_hot_mean(t) for t in X_train_txt])
X_test_oh = np.array([one_hot_mean(t) for t in X_test_txt])
print("X_train_oh:", X_train_oh.shape, " X_test_oh:", X_test_oh.shape)

label_of_word = {w: i for i, w in enumerate(vocab)}

def label_mean(text):
    ws = [w for w in nlp.preprocess(text) if w in label_of_word]
    if not ws:
        return 0.0
    return float(np.mean([label_of_word[w] for w in ws]))

X_train_lab = np.array([[label_mean(t)] for t in X_train_txt])
X_test_lab = np.array([[label_mean(t)] for t in X_test_txt])

print("\nEjemplo de One-Hot mean (primeros 10):")
print(np.round(X_train_oh[0, :10], 3))
print("Ejemplo de Labelling:")
print(np.round(X_train_lab[0], 3))

In [ ]:
def evaluate_model(name, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    return {
        "model": name,
        "accuracy": accuracy_score(yte, pred),
        "precision": precision_score(yte, pred, average="macro", zero_division=0),
        "recall": recall_score(yte, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(yte, pred, average="macro", zero_division=0),
    }

def fresh(name):
    if name.startswith("KNN"):
        return KNeighborsClassifier(n_neighbors=3)
    if name.startswith("LogisticRegression"):
        return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    if name.startswith("GaussianNB"):
        return GaussianNB()
    if name.startswith("DecisionTree"):
        return DecisionTreeClassifier(random_state=RANDOM_STATE)
    raise ValueError(name)

base_names = ["KNN(k=3)", "LogisticRegression", "GaussianNB", "DecisionTree"]

results = [evaluate_model(n, fresh(n), X_train_oh, y_train, X_test_oh, y_test) for n in base_names]
df_results = pd.DataFrame(results).sort_values("f1_macro", ascending=False).reset_index(drop=True)

print("===== Comparativa de modelos (One-Hot mean) =====")
print(df_results.round(4).to_string(index=False))

lab_results = [evaluate_model(f"{n} [label]", fresh(n), X_train_lab, y_train, X_test_lab, y_test) for n in base_names]
print("\n===== Comparativa con Labelling (baseline) =====")
print(pd.DataFrame(lab_results).round(4).to_string(index=False))

best_name = df_results.iloc[0]["model"]
best_model = fresh(best_name)
best_model.fit(X_train_oh, y_train)
y_pred = best_model.predict(X_test_oh)

print(f"\n===== Mejor modelo: {best_name} =====")
print("\nClassification report:")
print(classification_report(y_test, y_pred))

labels = sorted(set(y_test.tolist()) | set(y_pred.tolist()))
cm = confusion_matrix(y_test, y_pred, labels=labels)
print("Matriz de confusión:")
print(pd.DataFrame(cm, index=labels, columns=labels))

print("\n===== KFold (n_splits=5) con el mejor modelo =====")
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_all_oh = np.concatenate([X_train_oh, X_test_oh])
y_all = np.concatenate([y_train, y_test])

fold_scores = []
for i, (tr, te) in enumerate(kf.split(X_all_oh), start=1):
    fold_model = fresh(best_name)
    fold_model.fit(X_all_oh[tr], y_all[tr])
    pred = fold_model.predict(X_all_oh[te])
    fold_scores.append({
        "fold": i,
        "accuracy": accuracy_score(y_all[te], pred),
        "f1_macro": f1_score(y_all[te], pred, average="macro", zero_division=0),
    })
    print(f"  Fold {i}: acc={fold_scores[-1]['accuracy']:.4f}  f1_macro={fold_scores[-1]['f1_macro']:.4f}")

df_folds = pd.DataFrame(fold_scores)
print("\nMedia ± std:")
print(df_folds[["accuracy", "f1_macro"]].agg(["mean", "std"]).round(4))